# House Prices — V8: Comparable-Sale Adjustment

**Recorded Kaggle log RMSE** 0.11101

This notebook recreates the scored submission from saved model predictions and the repository data. Run the cells in order from a Python kernel with the packages in [final_versions/requirements.txt](final_versions/requirements.txt) installed.

## 1. Load the competition data

In [1]:
from io import BytesIO
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

current = Path.cwd()
root = next(
    (path for path in (current, *current.parents)
     if (path / "final_versions" / "rebuild.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook from the project repository.")
sys.path.insert(0, str(root))
from final_versions import rebuild

data = root / "Dataset"
if not data.is_dir():
    data = root / "house-prices-advanced-regression-techniques"
test = pd.read_csv(data / "test.csv", keep_default_na=False, na_values=["", "NA"])
assert len(test) == 1459 and test["Id"].is_unique

version = "V8"

## 2. Build the predictions

V8 starts from V7. It searches the training data for comparable large
new homes marked `Partial` and uses similar neighborhood, building type,
quality, living area and lot area. The final price gives equal weight
to the V7 estimate and the median comparable-sale price on the log scale.

In [2]:
train = pd.read_csv(
    data / "train.csv", keep_default_na=False, na_values=["", "NA"]
)
v7 = (rebuild.SUBMISSIONS / "v7.csv").read_bytes()
payload = rebuild.rebuild_v8(train, test, v7)
comparison = pd.DataFrame({
    "Id": test["Id"],
    "V7 estimate": rebuild.read_prices(v7),
    "V8 estimate": rebuild.read_prices(payload),
})
changed = ~np.isclose(
    comparison["V7 estimate"], comparison["V8 estimate"],
    rtol=1e-12, atol=1e-7,
)
display(comparison.loc[changed].round(2))

,Id,V7 estimate,V8 estimate
1089,2550,339331.06,241539.36


## 3. Verify and save the submission

In [3]:
rebuild.compare_with_submission(version, payload, test["Id"])
output = root / "rebuilt" / rebuild.VERSION_FILES[version]
output.parent.mkdir(parents=True, exist_ok=True)
output.write_bytes(payload)
submission = pd.read_csv(output)
scored = rebuild.SUBMISSIONS / output.name
display(pd.DataFrame([{
    "Output": str(output.relative_to(root)),
    "Rows": len(submission),
    "Comparison with scored CSV": (
        "Byte-identical" if payload == scored.read_bytes() else "Numerically matched"
    ),
}]))
display(submission.head(5).round(2))

,Output,Rows,Comparison with scored CSV
0,rebuilt/v8.csv,1459,Byte-identical


,Id,SalePrice
0,1461,122413.87
1,1462,161599.65
2,1463,187559.05
3,1464,198818.62
4,1465,182330.24


## Results

One test home matched the rule. Its estimate moved from **339,331.06**
to **241,539.36**. The recorded Kaggle log RMSE was **0.11101**.